In [ ]:
# custom imports
import nekt
import os                           # remove to production
from dotenv import load_dotenv      # remove to production
from pathlib import Path
from typing import Optional 
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# get Nekt data access token        # remove to production
load_dotenv()
nekt.data_access_token = os.getenv("NEKT_DATA_ACCESS_TOKEN")

# auxiliary functions
def load_nekt_table(table_name: str, layer_name: str) -> DataFrame:
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def display_df(df: DataFrame, limit=10):
    return df.limit(limit).toPandas()

def save_nekt_table(
    df: DataFrame, 
    layer_name: str, 
    table_name: str,
    folder_name: Optional[str] = None 
):
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )

# process:
##  load source tables
### clickup tables
df_bronze_users        = load_nekt_table("users"        , "Bronze")
df_bronze_spaces       = load_nekt_table("spaces"       , "Bronze")
df_bronze_time_entries = load_nekt_table("time_entries" , "Bronze")

### contaazul tables


## filter columns
df_silver_users = df_bronze_users.select(
      F.col("id").cast("integer").alias("user_id")
    , F.col("username").alias("user_name")
).filter(
    F.col("user_name").isNotNull()
).dropDuplicates(
    ["user_id"]
)

df_silver_spaces = df_bronze_spaces.select(
      F.col("id").cast("long").alias("space_id")
    , F.col("name").alias("space_name")
).filter(
    F.col("space_name").isNotNull()
).dropDuplicates(
    ["space_id"]
)

df_silver_time_entries = df_bronze_time_entries.select(
      F.col("id").cast("long").alias("interval_id")
    , F.col("wid").cast("integer").alias("team_id")
    , F.col("user.id").cast("integer").alias("user_id")
    , F.col("user.username").alias("user_name")
    , F.col("task_location.space_id").cast("long").alias("space_id")
    , F.col("task.id").alias("task_id")
    , F.col("start").cast("long").alias("interval_date_start_ms")
    , F.to_timestamp(F.col("start") / 1000).alias("interval_date_start_iso")
    , F.col("end").cast("long").alias("interval_date_end_ms")
    , F.to_timestamp(F.col("end") / 1000).alias("interval_date_end_iso")
    , F.col("at").cast("long").alias("interval_date_added_ms")
    , F.to_timestamp(F.col("at") / 1000).alias("interval_date_added_iso")
).filter(
      F.col("interval_id").isNotNull()
    & F.col("user_id").isNotNull()
    & F.col("interval_date_start_ms").isNotNull()
    & F.col("interval_date_end_ms").isNotNull()
    & (F.col("interval_date_end_ms") > F.col("interval_date_start_ms"))  # validating business rule
).dropDuplicates(
    ["interval_id"]
)

## save results to output tables
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

: 